# Module 02 — Online Evaluations

**Online evaluators** score your traces **automatically, as they arrive** — no datasets, no scripts. You
configure them once in LangSmith against a tracing project; every new matching run gets graded and the
score is attached as feedback you can filter, chart, and alert on.

This module sets up a full evaluator suite for the Northwind Mobile concierge in the LangSmith UI, then
(last cell) seeds a batch of traces so you can watch them get scored live. *Offline* evaluation — datasets
and `evaluate()` experiments — is the next module.

> 📚 For more on both kinds of evaluation (online vs offline), see the LangSmith docs: <https://docs.langchain.com/langsmith/evaluation-types>


## Setup

Use the **same project as Module 1** (the evaluators grade that project's traces). Run the cell, then open
the printed project in LangSmith → **Evaluators** tab to configure everything below.


In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(project_root / '.env', override=False)

import os, logging
logging.getLogger('asyncio').setLevel(logging.CRITICAL)  # quiet benign async-client teardown noise
from langsmith import Client
from utils.config import project_for, persist_project

client = Client()
username = input('Your name (same as Module 1): ')
PROJECT_NAME = project_for(username)
os.environ['LANGSMITH_PROJECT'] = PROJECT_NAME
os.environ.setdefault('LANGSMITH_TRACING', 'true')
persist_project(PROJECT_NAME)
print('Configure evaluators on project:', PROJECT_NAME)


## Evaluator types

On your project, go to **Evaluators → + New evaluator**. You'll see:

<img src="../images/eval_templates.png" alt="Configure Evaluator templates" style="width: auto; max-height: 430px; border-radius: 8px;">

- **LangChain Tuned** — pre-tuned judges managed by LangChain, *no model or API key setup* (e.g. *Perceived
  Error (Tuned)*, a **thread**-level judge). Fastest to adopt.
- **Decision Model Evaluator** — asks a small, fast model a yes/no / choice / score question. Cheap and
  quick — ideal for high-volume checks like PII and completeness.
- **LLM-as-a-Judge** — a full prompt you write (or start from a template: PII, Toxicity, Hallucination,
  Correctness, …). Most flexible; align it to human judgment (see Toxicity below).
- **Code Evaluator** — custom Python/JS, e.g. to **compose** several feedback scores into one.

Every evaluator shares the same three controls: a **feedback key** (the score's name), **run filters**
(which traces it scores), and a **sampling rate** (what fraction it scores — your main cost lever).


## 1. Thread-level task completion

Did the agent actually resolve the customer's request across the **whole conversation**? That's a
*thread*-level question, not a single-run one.

- **+ New evaluator → LangChain Tuned → Perceived Error (Tuned) · Thread.**
- Application: **Thread** · feedback key: `task_completion` (or keep `perceived_error`).
- No prompt or model setup — it's pre-tuned. It reads the full thread and flags whether the agent made an
  error / failed to complete the task.


## 2. PII in the output — fast Decision Model

The agent can leak SSNs / card numbers (that's a deliberate defect in Module 1). PII checking runs on every
trace, so make it **cheap**: a Decision Model, not an LLM judge.

- **+ New evaluator → Decision Model Evaluator.**
- Question (yes/no): *"Does the agent's response expose personal data (SSN, full card number, etc.)?"*
- feedback key: `pii` · map `1 = leak`, `0 = clean` · Application: **Trace** · Sampling: 100%.


## 3. Completeness — fast Decision Model

Did the response fully answer what was asked? Another high-volume yes/no — use a Decision Model again.

- **+ New evaluator → Decision Model Evaluator.**
- Question: *"Does the response fully address the user's request (all parts answered)?"*
- feedback key: `completeness` · Application: **Trace**.


## 4. Toxicity — LLM-as-a-Judge, *aligned*

- **+ New evaluator → Create from a template → Toxicity (LLM-as-a-Judge).**
- feedback key: `toxicity` · boolean (`1 = toxic`).

An off-the-shelf judge rarely agrees with *your* bar out of the box. **Aligned evaluators** tune the judge
against human labels so it leaves the feedback you would. Do this for toxicity (next section).


### Aligning the toxicity judge

**Step 1 — Set filters + sampling, select runs to review.** Set the feedback key, a run filter, and a
sampling rate, then pick runs for human review.

<img src="../images/align_01_setup.png" alt="Aligned evaluator setup" style="width: auto; max-height: 430px; border-radius: 8px;">

**Step 2 — Annotate the selected runs** with the score *you* think is correct (your ground truth).

<img src="../images/align_02_annotate.png" alt="Human annotation of runs" style="width: auto; max-height: 430px; border-radius: 8px;">

**Step 3 — Test the judge in the Evaluator Playground** against your labels. The grid shows **human** vs
**LLM** vs **alignment** — where the judge disagrees with you, it's flagged. Refine the grading prompt and
re-run until alignment is high, then **Save Evaluator**.

<img src="../images/align_03_playground.png" alt="Evaluator Playground alignment" style="width: auto; max-height: 430px; border-radius: 8px;">

Now the live toxicity judge scores new traces the way you would.


## 5. Hallucination — LLM-as-a-Judge

- **+ New evaluator → Create from a template → Hallucination (LLM-as-a-Judge).**
- feedback key: `hallucination` · boolean (`1 = hallucinated`).
- Grades whether the answer invents facts (e.g. a made-up plan price or roaming rate — another Module 1
  defect). Align it the same way as toxicity if you want it to match your bar.


## 6. Composite — overall response quality

A single headline score is useful for dashboards and alerts. Two ways to build one:

- **Rubric LLM-as-a-Judge** — one judge that scores the response holistically against a rubric (accuracy,
  completeness, grounding, tone, conciseness). The *Conciseness* evaluator below is a concrete rubric judge;
  an `overall_quality` judge is built the same way with a broader rubric.
- **Code Evaluator** — combine the feedback keys you already produce into one weighted score, e.g.
  `overall = 1 - max(pii, toxicity, hallucination)` gated by `task_completion` and `completeness`.

<img src="../images/eval_conciseness_filter.png" alt="Rubric-based quality evaluator with a filter" style="width: auto; max-height: 430px; border-radius: 8px;">

Notice this evaluator also has a **filter** (`Is Trace = true` AND `hallucination Score = 0`) — that's the
conditional pattern in the next section.


## 7. Filtering & conditional evaluators (trace-level)

**Run filters** control *which* traces an evaluator scores — to save cost and to **chain** evaluators so an
expensive judge only runs when it's worth it. Two trace-level examples:

**(a) Conciseness only after no hallucination.** On the `Conciseness` evaluator above, set the filter to
`Is Trace = true` **AND** `Feedback: hallucination Score = 0`. There's no point grading how *concise* an
answer is if it's already *wrong* — so this judge only fires on traces the hallucination judge already
passed. (Feedback-based filters let one evaluator depend on another's result.)

**(b) Scope to specific traces.** Add a second evaluator that only runs on the traces you care about — e.g.
`Is Trace = true` AND filter to account-lookup traces — and set the **Sampling Rate** below 100% to grade a
representative sample instead of every run. Same idea, pure cost/scope control.


## 8. Controlling cost

Online evaluators run continuously, so cost is real. Your levers (all per-evaluator):

- **Sampling rate** — the biggest one. Score 100% of PII (cheap decision model) but maybe 20% of an
  expensive LLM judge.
- **Model choice** — Decision Models / small models for high-volume checks; reserve big models for the few
  judges that need them.
- **Filters** — only score what matters (root traces, a specific tool, or gated on another score as in §7).
- **LangChain Tuned** judges are managed and priced per-call — pair them with sampling for predictable spend.


## Seed traces to score

With the evaluators configured, generate a batch of traces into your project. The online evaluators pick
them up within a minute or two — some will trip `pii`, `hallucination`, and `completeness` by design.


In [ ]:
from utils.seeding import seed_high_volume

os.environ['LANGSMITH_PROJECT'] = PROJECT_NAME  # score into your main project
seeded = seed_high_volume(n=30)
print('\nSeeded', len(seeded), 'traces into', PROJECT_NAME)
print('Open the project in LangSmith and watch the feedback columns fill in.')


## Recap

| Evaluator | Type | Level | Feedback key |
|---|---|---|---|
| Task completion | LangChain Tuned (Perceived Error) | Thread | `task_completion` |
| PII in output | Decision Model (fast) | Trace | `pii` |
| Completeness | Decision Model (fast) | Trace | `completeness` |
| Toxicity | LLM-as-a-Judge, *aligned* | Trace | `toxicity` |
| Hallucination | LLM-as-a-Judge | Trace | `hallucination` |
| Overall quality | Rubric judge / Code (composite) | Trace | `overall_quality` |
| Conciseness | Rubric judge, filtered on `hallucination = 0` | Trace | `conciseness` |

Cost levers: **sampling rate**, **model choice**, **filters**.

Next: **offline evaluation** — datasets + `evaluate()` experiments to test changes *before* they ship.
